In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp "/content/drive/MyDrive/Datasets/IEOR4578_Forecasting/master_long_hourly_test_2014_05_12.csv" "/content/master_long_hourly_test_2014_05_12.csv"
!cp "/content/drive/MyDrive/Datasets/IEOR4578_Forecasting/master_long_hourly_train_2012_2013.csv" "/content/master_long_hourly_train_2012_2013.csv"
!cp "/content/drive/MyDrive/Datasets/IEOR4578_Forecasting/master_long_hourly_validation_2014_01_04.csv" "/content/master_long_hourly_validation_2014_01_04.csv"

In [3]:
!pip install statsforecast

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.6/354.6 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 6.1 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [4]:
from pathlib import Path

# Input files — produced by preprocessing + data-split teammate
TRAIN_PATH    = "/content/master_long_hourly_train_2012_2013.csv"
VAL_PATH      = "/content/master_long_hourly_validation_2014_01_04.csv"
TEST_PATH     = "/content/master_long_hourly_test_2014_05_12.csv"
CALENDAR_PATH = "/content/calendar_features_hourly.csv"

# Output — share this file with teammates for alignment
SELECTED_CLIENTS_PATH = Path("selected_clients_autoarima.csv")

# Sampling
RANDOM_SEED = 42
# N_CLIENTS   = 50    # increase carefully — SARIMAX is slow per client

# Model
SEASON_LENGTH = 24  # hourly data → daily seasonality
LOOKBACK_HOURS = 672 # 4 weeks lookback
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

In [5]:
# IMPORTS
import warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import ARIMA, AutoARIMA

In [6]:
# ── SAMPLE CLIENTS ───────────────────────────────────────────────────────────
# Read only the client_id column to avoid loading the full file
train_ids = pd.read_csv(TRAIN_PATH, usecols=["client_id"])
all_clients = train_ids["client_id"].unique()

# rng = np.random.default_rng(RANDOM_SEED)
# selected = sorted(rng.choice(all_clients, size=N_CLIENTS, replace=False).tolist())
selected = sorted(all_clients.tolist()) # Use all clients

print(f"Selected {len(selected)} clients (ALL)")

# Save so teammates can use the same clients
pd.DataFrame({"client_id": selected}).to_csv(SELECTED_CLIENTS_PATH, index=False)
print(f"Saved client list → {SELECTED_CLIENTS_PATH}")

Selected 156 clients (ALL)
Saved client list → selected_clients_autoarima.csv


In [7]:
# ── PREPARE DATA ─────────────────────────────────────────────────────────────
# Removed exogenous variables to convert to pure AutoARIMA.
# EXOG_COLS = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend", "month_sin", "month_cos"]

# calendar = pd.read_csv(CALENDAR_PATH, parse_dates=["timestamp"])


def load_split(path, clients):
    df = pd.read_csv(path, parse_dates=["timestamp"])
    df = df[df["client_id"].isin(clients)].copy()
    # df = df.merge(calendar[["timestamp"] + exog_cols], on="timestamp", how="left") # Removed merge with calendar
    # StatsForecast expects: unique_id, ds, y
    df = df.rename(columns={"client_id": "unique_id", "timestamp": "ds"})
    return df.sort_values(["unique_id", "ds"]).reset_index(drop=True)


def apply_lookback(df, hours):
    """Keep only the last `hours` rows per client."""
    return (df.groupby("unique_id", group_keys=False)
              .tail(hours)
              .reset_index(drop=True))


train = load_split(TRAIN_PATH, selected)
val   = load_split(VAL_PATH,   selected)
test  = load_split(TEST_PATH,  selected)

# Apply 4-week lookback: train on the most recent 672 hours per client
train = apply_lookback(train, LOOKBACK_HOURS)

print(f"train : {train.shape}  {train['ds'].min()} → {train['ds'].max()}")
print(f"val   : {val.shape}    {val['ds'].min()} → {val['ds'].max()}")
print(f"test  : {test.shape}   {test['ds'].min()} → {test['ds'].max()}")

train : (104832, 3)  2013-12-04 00:00:00 → 2013-12-31 23:00:00
val   : (445536, 3)    2014-01-01 00:00:00 → 2014-04-30 23:00:00
test  : (913536, 3)   2014-05-01 00:00:00 → 2014-12-31 23:00:00


In [8]:
# ── EVALUATION METRICS ────────────────────────────────────────────────────────
def compute_metrics(df, target_col="y", pred_col="ARIMA"):
    y    = df[target_col].values
    yhat = df[pred_col].values
    mse  = np.mean((y - yhat) ** 2)
    mae  = np.mean(np.abs(y - yhat))
    wape = np.sum(np.abs(y - yhat)) / np.sum(np.abs(y))
    return {"MSE": round(mse, 4), "MAE": round(mae, 4), "WAPE": round(wape, 4)}


def per_client_metrics(df, target_col="y", pred_col="ARIMA"):
    rows = []
    for uid, grp in df.groupby("unique_id"):
        rows.append({"client_id": uid, **compute_metrics(grp, target_col, pred_col)})
    rows.append({"client_id": "OVERALL", **compute_metrics(df, target_col, pred_col)})
    return pd.DataFrame(rows)

In [9]:
# ── TRAIN AutoARIMA ─────────────────────────────────────────────────────────────
# Switching to AutoARIMA allows the model to select the best parameters (p,d,q)
# automatically for each client, preventing the poor fit of a fixed order.
model = AutoARIMA(
    season_length=SEASON_LENGTH,
    approximation=True,  # Faster search
    stepwise=True        # Faster search
)

sf = StatsForecast(models=[model], freq="h", n_jobs=-1)
sf.fit(train[["unique_id", "ds", "y"]])

joblib.dump(sf, MODEL_DIR / "autoarima_val.joblib")
print("Training complete. Model saved → models/autoarima_val.joblib")

Training complete. Model saved → models/autoarima_val.joblib


In [10]:
# ── VALIDATION FORECAST ───────────────────────────────────────────────────────
# h = number of unique timestamps in val (same for all clients after a clean time-based split)
val_h = val["ds"].nunique()
# X_val = val[["unique_id", "ds"] + EXOG_COLS] # Removed exogenous variables

val_preds = sf.predict(h=val_h)
val_eval  = val[["unique_id", "ds", "y"]].merge(val_preds, on=["unique_id", "ds"])

print(val_eval.head())

  unique_id                  ds          y   AutoARIMA
0    MT_124 2014-01-01 00:00:00  95.693780  135.397727
1    MT_124 2014-01-01 01:00:00  83.732056  126.806499
2    MT_124 2014-01-01 02:00:00  82.535890  111.681575
3    MT_124 2014-01-01 03:00:00  82.535890   80.470334
4    MT_124 2014-01-01 04:00:00  63.397133   40.219741


In [11]:
# 1. Training Metrics (In-Sample)
# Access fitted models directly and call predict_in_sample()
train_preds_list = []
for i, uid in enumerate(train["unique_id"].unique()):
    fitted_model = sf.fitted_[i, 0]  # [series_index, model_index]
    client_train = train[train["unique_id"] == uid].copy()

    # Get in-sample predictions
    in_sample_preds = fitted_model.predict_in_sample()

    # Check if it's a dict and extract the mean, otherwise use directly
    if isinstance(in_sample_preds, dict):
        preds = in_sample_preds.get("mean", in_sample_preds.get("fitted", list(in_sample_preds.values())[0]))
    else:
        preds = in_sample_preds

    # Create dataframe with predictions
    pred_df = pd.DataFrame({
        "unique_id": [uid] * len(client_train),
        "ds": client_train["ds"].values,
        "AutoARIMA": preds
    })
    train_preds_list.append(pred_df)

train_preds = pd.concat(train_preds_list, ignore_index=True)
train_eval = train[["unique_id", "ds", "y"]].merge(train_preds, on=["unique_id", "ds"], how="left")

train_metrics = per_client_metrics(train_eval.dropna(), pred_col="AutoARIMA")
print("── Training Metrics (In-Sample) ──")
# Show first 5 and OVERALL
print(train_metrics.head(5).to_string(index=False))
print("...")
print(train_metrics.tail(1).to_string(index=False))
print("\n")

# 2. Validation Metrics (Out-of-Sample)
val_metrics = per_client_metrics(val_eval, pred_col="AutoARIMA")
print("── Validation Metrics (Out-of-Sample) ──")
# Show first 5 and OVERALL
print(val_metrics.head(5).to_string(index=False))
print("...")
print(val_metrics.tail(1).to_string(index=False))

── Training Metrics (In-Sample) ──
client_id      MSE     MAE   WAPE
   MT_124 937.7737 22.1754 0.0828
   MT_132  79.9288  4.8413 0.2980
   MT_156  68.4093  5.6308 0.0839
   MT_158 350.8809 12.6300 0.1698
   MT_159 178.2141  8.0367 0.1309
...
client_id        MSE     MAE   WAPE
  OVERALL 15165.6659 26.8957 0.0432


── Validation Metrics (Out-of-Sample) ──
client_id       MSE     MAE   WAPE
   MT_124 9758.8758 77.7164 0.2870
   MT_132  225.7467 12.1202 0.7088
   MT_156 1337.6730 30.0115 0.3676
   MT_158 1862.8863 37.7304 0.5113
   MT_159 1774.7561 39.3452 0.7728
...
client_id         MSE     MAE   WAPE
  OVERALL 111368.7255 86.8246 0.1416


In [12]:
# ── TEST EVALUATION ───────────────────────────────────────────────────────────
# Retrain on train+val (with lookback), then predict on the held-out test set
train_val = pd.concat([train, val], ignore_index=True).sort_values(["unique_id", "ds"])
train_val = apply_lookback(train_val, LOOKBACK_HOURS)

# Redefine the model here to ensure it's available
model = AutoARIMA(
    season_length=SEASON_LENGTH,
    approximation=True,  # Faster search
    stepwise=True        # Faster search
)

sf_final = StatsForecast(models=[model], freq="h", n_jobs=-1)
sf_final.fit(train_val[["unique_id", "ds", "y"]])

joblib.dump(sf_final, MODEL_DIR / "autoarima_final.joblib")
print("Final model saved → models/autoarima_final.joblib")

test_h     = test["ds"].nunique()
# X_test     = test[["unique_id", "ds"] + EXOG_COLS] # Removed exogenous variables
test_preds = sf_final.predict(h=test_h)
test_eval  = test[["unique_id", "ds", "y"]].merge(test_preds, on=["unique_id", "ds"])

test_metrics = per_client_metrics(test_eval, pred_col="AutoARIMA")
print("── Test metrics ──")
print(test_metrics.to_string(index=False))

Final model saved → models/autoarima_final.joblib
── Test metrics ──
client_id          MSE       MAE   WAPE
   MT_124 5.004056e+03   51.5774 0.1717
   MT_132 2.282095e+02   14.1171 0.9328
   MT_156 6.038655e+02   17.8161 0.2003
   MT_158 1.853621e+03   34.8743 0.4449
   MT_159 1.510147e+03   28.6541 0.6692
   MT_161 9.018640e+04  253.9564 0.1620
   MT_162 1.479154e+04   95.9526 0.3539
   MT_163 1.498655e+05  293.0747 0.1172
   MT_166 1.844580e+05  335.7546 0.2599
   MT_168 1.656341e+02    9.8378 0.0735
   MT_169 3.463340e+01    4.8980 0.1162
   MT_171 5.618941e+03   59.6717 0.2526
   MT_172 3.428421e+02   13.8146 0.1130
   MT_174 7.948171e+02   19.1385 0.1165
   MT_175 4.986271e+02   15.6181 0.0982
   MT_176 1.981618e+02   10.3722 0.1202
   MT_180 7.754551e+02   20.8001 0.1392
   MT_182 4.773910e+01    5.0120 0.0996
   MT_183 1.690110e+02    9.1966 0.1188
   MT_187 3.356677e+02   14.1759 0.1526
   MT_188 2.781580e+01    3.7110 0.0875
   MT_189 3.989147e+03   46.0132 0.1150
   MT_190 2